In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip uninstall -y transformers accelerate datasets sentence-transformers tensorflow jax jaxlib
!pip install transformers==4.30.2 accelerate==0.20.3 datasets==2.13.0 sentence-transformers==2.2.2 tensorflow==2.13.0
!pip install jiwer gradio typing-extensions
!pip install chromadb sentence-transformers rank_bm25 numpy py_vncorenlp rouge_score bert_score vncorenlp evaluate
!pip install huggingface_hub==0.16.4

In [ ]:
import py_vncorenlp

py_vncorenlp.download_model(save_dir='/content/vncorenlp',)

nlp = py_vncorenlp.VnCoreNLP(save_dir='/content/vncorenlp')



In [ ]:
from vncorenlp import VnCoreNLP
from collections import Counter
import math
from nltk.translate.bleu_score import SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
import random

In [ ]:
class VNPreprocessor:
    def __init__(self, nlp):
        self.nlp = nlp

    def preprocess(self, text):
        segmented_sentences = self.nlp.word_segment(text)
        processed_text = ' '.join(segmented_sentences)
        return processed_text

preprocessor = VNPreprocessor(nlp)

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from tqdm import tqdm
import numpy as np
import torch

# Đường dẫn dữ liệu
train_path = "/content/drive/MyDrive/CS313/train_data.csv"
val_path = "/content/drive/MyDrive/CS313/valid_data.csv"
test_path = "/content/drive/MyDrive/CS313/test_data.csv"

# Đọc dữ liệu
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

"""
# Tiền xử lý dữ liệu
def preprocess_data(df):
    df['processed_question'] = df['question'].str.lower().apply(preprocessor.preprocess)
    df['processed_context'] = df['context'].str.lower().apply(preprocessor.preprocess)
    return df

df_train = preprocess_data(df_train)
df_val = preprocess_data(df_val)
df_test = preprocess_data(df_test)
"""

# Tiền xử lý dữ liệu
def preprocess_data(df):
    df['processed_question'] = df['question'].str.lower().apply(preprocessor.preprocess)
    df['processed_context'] = df['context'].str.lower().apply(preprocessor.preprocess)
    return df

df_train = preprocess_data(df_train)
df_val = preprocess_data(df_val)
df_test = preprocess_data(df_test)

# Tạo cặp positive và negative
def prepare_pairs(data):
    positive_pairs = []
    negative_pairs = []
    context_grouped = data.groupby('processed_context')

    for _, row in tqdm(data.iterrows(), desc="Creating pairs", total=len(data)):
        question = row['processed_question']
        correct_context = row['processed_context']

        # Positive pair
        positive_pairs.append(InputExample(texts=[question, correct_context], label=1))

        # Negative sampling (context không thuộc nhóm câu hỏi hiện tại)
        other_contexts = context_grouped.filter(lambda x: x.name != correct_context)['processed_context'].unique()
        if len(other_contexts) > 0:
            random_negative = np.random.choice(other_contexts, 1)[0]
            negative_pairs.append(InputExample(texts=[question, random_negative], label=0))

    return positive_pairs + negative_pairs

# Chuẩn bị dữ liệu train, val, test
train_data = prepare_pairs(df_train)
val_data = prepare_pairs(df_val)
test_data = prepare_pairs(df_test)

# DataLoader
train_dataloader = DataLoader(train_data, shuffle=True, batch_size=58)
val_dataloader = DataLoader(val_data, shuffle=False, batch_size=58)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=58)

# Khởi tạo Sentence-BERT
model_name = "bkai-foundation-models/vietnamese-bi-encoder"
sbert_model = SentenceTransformer(model_name)

# Loss function
loss = losses.MultipleNegativesRankingLoss(sbert_model)

# Fine-tuning
print("Fine-tuning Sentence-BERT...")
sbert_model.fit(
    train_objectives=[(train_dataloader, loss)],
    evaluator=val_dataloader,
    evaluation_steps=200,
    epochs=5,
    warmup_steps=100,
    output_path="fine_tuned_sbert_model",
)

# Đánh giá trên tập test với P@K
def evaluate_p_at_k(model, test_dataloader, ground_truth_dict, topk=1):
    model = SentenceTransformer("fine_tuned_sbert_model")
    all_questions = [example.texts[0] for example in test_dataloader.dataset]
    all_contexts = [example.texts[1] for example in test_dataloader.dataset]

    question_embeddings = model.encode(all_questions, convert_to_tensor=True)
    context_embeddings = model.encode(all_contexts, convert_to_tensor=True)

    # Tính cosine similarity
    cosine_scores = torch.mm(question_embeddings, context_embeddings.T)

    # Precision@K
    precision_at_k = []
    for i, question in enumerate(all_questions):
        ground_truth_contexts = ground_truth_dict.get(question, [])
        retrieved_indices = torch.topk(cosine_scores[i], k=topk).indices
        retrieved = [all_contexts[idx.item()] for idx in retrieved_indices]

        # Precision@k
        relevant_retrieved = sum(ctx in ground_truth_contexts for ctx in retrieved)
        precision_at_k.append(relevant_retrieved / topk)

    return {
        f"P@{topk}": sum(precision_at_k) / len(precision_at_k),
    }

# Tạo ground truth dictionary cho test
ground_truth_dict = df_test.groupby('processed_question')['processed_context'].apply(list).to_dict()

# Đánh giá kết quả
print("Evaluating on test data...")
metrics = evaluate_p_at_k(sbert_model, test_dataloader, ground_truth_dict, topk=1)
print(f"P@1: {metrics['P@1']}")


In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from tqdm import tqdm
import numpy as np
import torch
import os
import wandb
import random

def set_SEED():
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(SEED)

set_SEED()

# ==== CÀI ĐẶT WANDB ====
# WANDB_API_KEY = "9b9b0a2ebf5574038e6c07c37c0912a52b7fbce5"  # Thay thế bằng API key của bạn
os.environ["WANDB_API_KEY"] = WANDB_API_KEY  # Thiết lập biến môi trường
wandb.login(key=WANDB_API_KEY)  # Tự động đăng nhập wandb
wandb.init(project="finetune-embedding", name="sbert-finetuning")

# ==== THÔNG SỐ ====
BATCH_SIZE = 58
EPOCHS = 5
LEARNING_RATE = 1e-4
WARMUP_STEPS = 200
MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"

# ==== ĐƯỜNG DẪN DỮ LIỆU ====
data_dir = "/kaggle/input/dataset-finetune-embedding/"
train_path = os.path.join(data_dir, 'train_pairs.csv')
val_path = os.path.join(data_dir, 'val_pairs.csv')
test_path = os.path.join(data_dir, 'test_pairs.csv')

# ==== ĐỌC DỮ LIỆU ====
print("Đọc dữ liệu...")
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

# ==== CHUYỂN ĐỔI DỮ LIỆU ====
def prepare_pairs(df):
    pairs = []
    for _, row in df.iterrows():
        pairs.append(InputExample(texts=[row['question'], row['context']], label=row['label']))
    return pairs

train_data = prepare_pairs(df_train)
val_data = prepare_pairs(df_val)
test_data = prepare_pairs(df_test)

# ==== MODEL ====
print("Khởi tạo model...")
sbert_model = SentenceTransformer(MODEL_NAME)

train_dataloader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)
# ==== LOSS FUNCTION ====
loss = losses.MultipleNegativesRankingLoss(sbert_model)

# ==== EVALUATOR ====
val_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(val_data, name="val-eval")

# ==== TRAINING ====
print("Fine-tuning Sentence-BERT...")
sbert_model.fit(
    train_objectives=[(train_dataloader, loss)],
    evaluator=val_evaluator,  # Đúng kiểu dữ liệu
    epochs=EPOCHS,
    evaluation_steps=200,
    warmup_steps=WARMUP_STEPS,
    optimizer_params={'lr': LEARNING_RATE},
    output_path="fine_tuned_sbert_model"
)

# ==== ĐÁNH GIÁ ====
def evaluate_p_at_k(model, test_data, ground_truth_dict, topk=1):
    """
    Đánh giá kết quả trên Precision@K
    """
    model = SentenceTransformer("fine_tuned_sbert_model")
    all_questions = [example.texts[0] for example in test_data]
    all_contexts = [example.texts[1] for example in test_data]

    # Tính embeddings
    question_embeddings = model.encode(all_questions, convert_to_tensor=True)
    context_embeddings = model.encode(all_contexts, convert_to_tensor=True)

    # Tính cosine similarity
    cosine_scores = torch.mm(question_embeddings, context_embeddings.T)

    # Precision@K
    precision_at_k = []
    for i, question in enumerate(all_questions):
        ground_truth_contexts = ground_truth_dict.get(question, [])
        retrieved_indices = torch.topk(cosine_scores[i], k=topk).indices
        retrieved = [all_contexts[idx.item()] for idx in retrieved_indices]

        # Precision@k
        relevant_retrieved = sum(ctx in ground_truth_contexts for ctx in retrieved)
        precision_at_k.append(relevant_retrieved / topk)

    return {
        f"P@{topk}": sum(precision_at_k) / len(precision_at_k),
    }

# ==== ĐÁNH GIÁ TẬP TEST ====
print("Evaluating on test data...")
ground_truth_dict = df_test.groupby('question')['context'].apply(list).to_dict()
metrics = evaluate_p_at_k(sbert_model, test_data, ground_truth_dict, topk=1)
print(f"P@1: {metrics['P@1']:.4f}")


In [ ]:
save_path = "saved_model/sbert_model_"
os.makedirs(save_path, exist_ok=True)
sbert_model.save(save_path)
print(f"Model saved at {save_path}")

In [ ]:
import shutil

# Đường dẫn thư mục chứa model đã lưu
model_dir = "saved_model/sbert_model_"
zip_file = "fine_tuned_sbert_model_epoch_5.zip"

# Nén thư mục thành file zip
shutil.make_archive("fine_tuned_sbert_model_epoch_5", 'zip', model_dir)

# Xác nhận hoàn tất
zip_file


In [ ]:
# ==== ĐÁNH GIÁ ====
def evaluate_p_at_k(model, test_data, ground_truth_dict, topk=1):
    model = SentenceTransformer("fine_tuned_sbert_model")
    all_questions = [example.texts[0] for example in test_data]
    all_contexts = [example.texts[1] for example in test_data]

    # Tính embeddings
    question_embeddings = model.encode(all_questions, convert_to_tensor=True)
    context_embeddings = model.encode(all_contexts, convert_to_tensor=True)

    # Tính cosine similarity
    cosine_scores = torch.mm(question_embeddings, context_embeddings.T)

    # Precision@K
    precision_at_k = []
    for i, question in enumerate(all_questions):
        ground_truth_contexts = ground_truth_dict.get(question, [])
        retrieved_indices = torch.topk(cosine_scores[i], k=topk).indices
        retrieved = [all_contexts[idx.item()] for idx in retrieved_indices]

        print(f"Query: {question}")
        print(f"Retrieved: {retrieved}")

        # Precision@k
        relevant_retrieved = sum(ctx in ground_truth_contexts for ctx in retrieved)
        precision_at_k.append(relevant_retrieved / topk)

    return {
        f"P@{topk}": sum(precision_at_k) / len(precision_at_k),
    }

print("Evaluating on test data...")
ground_truth_dict = df_test.groupby('question')['context'].apply(list).to_dict()
metrics = evaluate_p_at_k(sbert_model, test_data, ground_truth_dict, topk=1)
print(f"P@1: {metrics['P@1']:.4f}")

# ==== KẾT QUẢ CUỐI CÙNG ====
print("Hoàn thành huấn luyện và lưu mô hình")